# 01 Data Preparation Validation

Purpose: Validate wide-to-tall conversion, blank handling, near-zero cleaning, and data quality checks.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

START_YEAR = 2025
END_YEAR = 2040
TOLERANCE = 1e-9
DATA_PATH = Path('../data/input_profiles.csv')

In [6]:
data_profile = pd.read_csv(DATA_PATH)
display(data_profile.head())

,ID,Case,Metric,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040
0,1,Project 1,Production,170,172,235,240,245,250,248,246,240,230,220,200,180,160,140,120
1,1,Project 1,Revenue Generating Production,150,152,210,215,220,225,223,220,215,205,195,180,160,140,120,100
2,1,Project 1,Liquids Production,90,91,126,129,132,135,134,132,129,123,117,108,96,84,72,60
3,1,Project 1,Gas Production,60,61,84,86,88,90,89,88,86,82,78,72,64,56,48,40
4,1,Project 1,Gross Margin,1515,1372,2144,2576,2827,2894,2861,2944,2896,2839,2506,2344,2053,1807,1628,1439


In [12]:
# =====================================================
# Data Profile: Wide to Tall Conversion
# =====================================================
# Excel version:
# In Python-in-Excel, the input profile table is read from the named range
# "inputs_profiles" using xl(). Excel sometimes passes arrays without
# reliable column headers, so headers are reassigned explicitly.
#
# data_profile = xl("inputs_profiles")
# headers = ["ID", "Case", "Metric"] + [str(y) for y in range(start_year, end_year + 1)]
# data_profile.columns = headers
#
# VS Code / notebook version:
# In the notebook, data_profile is loaded from CSV earlier using pd.read_csv().
# The same structure is expected:
# ID | Case | Metric | 2025 | 2026 | ... | 2040

year_cols = [str(y) for y in range(START_YEAR, END_YEAR + 1)]

# Convert from wide format:
# ID | Case | Metric | 2025 | 2026 | ...
#
# to tall format:
# ID | Case | Metric | Year | Value
data_profile_tall = data_profile.melt(
    id_vars=["ID", "Case", "Metric"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

# Year is stored as text after melt, so convert to integer for filtering,
# sorting, plotting, and scenario operations.
data_profile_tall["Year"] = data_profile_tall["Year"].astype(int)

# Convert values to numeric.
# Blank cells and invalid text become NaN, then are treated as zero because
# blanks represent inactive profile years in the planning model.
data_profile_tall["Value"] = pd.to_numeric(
    data_profile_tall["Value"],
    errors="coerce"
).fillna(0)

# Remove floating point source-system artefacts, e.g. -1.987654321E-300.
# These are treated as zero because they are not economically meaningful.
TOLERANCE = 1e-9
data_profile_tall.loc[
    data_profile_tall["Value"].abs() < TOLERANCE,
    "Value"
] = 0

# Round to a sensible precision to avoid unnecessary decimal noise.
data_profile_tall["Value"] = data_profile_tall["Value"].round(6)

# Remove inactive profile years.
# This reduces dataset size and prevents false activity being carried forward
# into scenario operations.
data_profile_tall = data_profile_tall[
    data_profile_tall["Value"] != 0
].copy()

print("\nSample from data_profile_tall:")
display(data_profile_tall.head())




Sample from data_profile_tall:


,ID,Case,Metric,Year,Value
0,1,Project 1,Production,2025,170
1,1,Project 1,Revenue Generating Production,2025,150
2,1,Project 1,Liquids Production,2025,90
3,1,Project 1,Gas Production,2025,60
4,1,Project 1,Gross Margin,2025,1515
